# Phase 1, Empirical Diagnostics

Goal: characterize the real S&P 500 return matrix so the downstream theory and algorithms are grounded.

**What this notebook computes, and which references each block draws on:**

| Block | Quantity | Reference |
|---|---|---|
| 1 | Per-stock sub-Gaussian norm $\|X\|_{\psi_2}$ | standard sub-Gaussian references |
| 2 | Per-stock sub-exponential norm $\|X\|_{\psi_1}$ | `L4 - subexponential.pdf`, Prop. 1, item 4 / Def. 1 |
| 3 | QQ plots vs Gaussian and Laplace | sanity check for items 1-2 |
| 4 | Empirical spectrum of $\hat\Sigma$ vs Marchenko-Pastur | standard random-matrix references (extension of Wigner to sample covariance) |
| 5 | Factor extraction; residual spectrum after removing top factors | motivates the dependent-Gaussian model used in OGP theory |
| 6 | Empirical RIP constant $\delta_k$ for $k\in\{5,10,20,50\}$ | standard compressive-sensing references + Thm. 1 |
| 7 | Date-stamped extreme-return events | motivates the time-revelation Doob martingale of standard concentration references |

Outputs go to `data/diagnostics_phase1.json` and `data/figs/`.

**Verdict at the bottom:** which regime we are in (sub-Gaussian / sub-exponential / heavier; weak / strong factor structure), this dictates the wording of the theory in Phases 3a and 4a.

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_DIR = Path("data")
FIG_DIR = DATA_DIR / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)

R_df = pd.read_parquet(DATA_DIR / "returns.parquet")
R = R_df.to_numpy()
T, n = R.shape
tickers = R_df.columns.to_numpy()
dates = R_df.index

mu = R.mean(axis=0)
sd = R.std(axis=0, ddof=1)
Z = (R - mu) / sd

print(f"Returns matrix:  T={T}  n={n}")
print(f"Date window:     {dates[0].date()} -> {dates[-1].date()}")
print(f"Mean daily mu:   {np.mean(mu)*1e4:+.1f} bp")
print(f"Mean daily sigma:{np.mean(sd)*1e2:+.2f}%")

## Block 1+2, Sub-Gaussian and sub-exponential norms

**Definition (Lec 3 / Lec 4, item 4 of the equivalent characterizations).**
$$
\|X\|_{\psi_2} \;=\; \inf\bigl\{t>0 : \mathbb{E}\,\exp(X^2/t^2) \le 2\bigr\},
\qquad
\|X\|_{\psi_1} \;=\; \inf\bigl\{t>0 : \mathbb{E}\,\exp(|X|/t) \le 2\bigr\}.
$$
We estimate each by binary search using the empirical mean. We use the log-sum-exp trick to avoid overflow on heavy-tailed columns. Reference values for calibration:
- $X\sim\mathcal N(0,1)$: $\|X\|_{\psi_2}\approx \sqrt{8/3}\approx 1.633$.
- $X\sim\mathrm{Laplace}(0,1)$ (Var=2), standardized: $\|X\|_{\psi_2}=\infty$, $\|X\|_{\psi_1}\approx 1/\ln 2 \approx 1.443$.

If many stocks have **finite** $\psi_2$ near the Gaussian reference, the sub-Gaussian theory (Hoeffding) is fine. If $\psi_2$ blows up but $\psi_1$ stays bounded, we are in the **sub-exponential regime** and must use Bernstein.

In [ ]:
def _log_mean_exp(vals: np.ndarray) -> float:
    m = vals.max()
    return m + np.log(np.mean(np.exp(vals - m)))


def estimate_psi(x: np.ndarray, kind: str, t_lo: float = 1e-2, t_hi: float = 50.0, tol: float = 1e-3) -> float:
    """Binary-search the smallest t > 0 with E exp(g(x)/t^?) <= 2.
    
    kind='psi2': condition is mean exp(x^2/t^2) <= 2
    kind='psi1': condition is mean exp(|x|/t) <= 2
    Returns t_hi if no t in [t_lo,t_hi] works (treat as 'infinity').
    """
    def feasible(t):
        if kind == "psi2":
            vals = (x / t) ** 2
        else:
            vals = np.abs(x) / t
        try:
            return _log_mean_exp(vals) <= np.log(2.0)
        except (OverflowError, FloatingPointError):
            return False

    if not feasible(t_hi):
        return np.inf
    while t_hi - t_lo > tol:
        mid = 0.5 * (t_lo + t_hi)
        if feasible(mid):
            t_hi = mid
        else:
            t_lo = mid
    return t_hi


psi2 = np.array([estimate_psi(Z[:, i], "psi2") for i in range(n)])
psi1 = np.array([estimate_psi(Z[:, i], "psi1") for i in range(n)])

psi2_finite = np.isfinite(psi2)
psi1_finite = np.isfinite(psi1)

print(f"ψ_2 finite:  {psi2_finite.sum()} / {n}  ({psi2_finite.mean()*100:.1f}%)")
print(f"ψ_1 finite:  {psi1_finite.sum()} / {n}  ({psi1_finite.mean()*100:.1f}%)")
if psi2_finite.any():
    print(f"ψ_2 quartiles (finite): {np.quantile(psi2[psi2_finite], [0.25,0.5,0.75])}")
if psi1_finite.any():
    print(f"ψ_1 quartiles (finite): {np.quantile(psi1[psi1_finite], [0.25,0.5,0.75])}")
print(f"Gaussian ref ψ_2: {np.sqrt(8/3):.3f}")
print(f"Laplace  ref ψ_1: {1/np.log(2):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

psi2_plot = np.where(np.isfinite(psi2), psi2, np.nan)
axes[0].hist(psi2_plot[~np.isnan(psi2_plot)], bins=40, color="steelblue", alpha=0.85)
axes[0].axvline(np.sqrt(8/3), color="red", lw=2, label=f"N(0,1) ref = {np.sqrt(8/3):.2f}")
n_inf2 = (~psi2_finite).sum()
axes[0].set_title(rf"$\psi_2$ norms (finite: {psi2_finite.sum()}/{n}; $\infty$: {n_inf2})")
axes[0].set_xlabel(r"$\|X\|_{\psi_2}$")
axes[0].legend()

axes[1].hist(psi1[psi1_finite], bins=40, color="darkorange", alpha=0.85)
axes[1].axvline(1/np.log(2), color="red", lw=2, label=f"Laplace ref = {1/np.log(2):.2f}")
n_inf1 = (~psi1_finite).sum()
axes[1].set_title(rf"$\psi_1$ norms (finite: {psi1_finite.sum()}/{n}; $\infty$: {n_inf1})")
axes[1].set_xlabel(r"$\|X\|_{\psi_1}$")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "psi_norms.png", dpi=120)
plt.show()

## Block 3, QQ plots

Visual confirmation. Pick three representative stocks: smallest, median, and largest $\psi_2$. Compare standardized returns against $\mathcal N(0,1)$ and Laplace.

In [ ]:
psi2_for_sort = np.where(np.isfinite(psi2), psi2, 1e6)
order = np.argsort(psi2_for_sort)
picks = [order[0], order[len(order) // 2], order[-1]]
labels = ["lightest tail", "median tail", "heaviest tail"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for j, (idx, lab) in enumerate(zip(picks, labels)):
    z = Z[:, idx]
    name = tickers[idx]
    p2 = psi2[idx]
    stats.probplot(z, dist="norm", plot=axes[0, j])
    axes[0, j].set_title(f"{name}  ({lab})  ψ_2={p2:.2f}\nQQ vs N(0,1)")
    stats.probplot(z, dist=stats.laplace(scale=1/np.sqrt(2)), plot=axes[1, j])
    axes[1, j].set_title("QQ vs standardized Laplace")
plt.tight_layout()
plt.savefig(FIG_DIR / "qq_plots.png", dpi=120)
plt.show()

## Block 4, Empirical spectrum of $\hat\Sigma$ vs Marchenko-Pastur

The standard standard random-matrix references treats Wigner symmetric matrices. The right reference for a sample-covariance matrix $\hat\Sigma = \frac{1}{T} R^\top R$ from i.i.d. columns of variance $\sigma^2$ is the **Marchenko-Pastur distribution** with aspect ratio $c = n/T$:
$$
f_{MP}(\lambda) = \frac{1}{2\pi\sigma^2 c\,\lambda}\sqrt{(\lambda_+ - \lambda)(\lambda - \lambda_-)},\quad \lambda_\pm = \sigma^2(1\pm\sqrt c)^2.
$$
Eigenvalues outside $[\lambda_-,\lambda_+]$ are the **factor / structural** modes, they are what makes real returns *not* a pure random matrix.

In [ ]:
Sigma_hat = np.cov(R, rowvar=False)
eigs = np.linalg.eigvalsh(Sigma_hat)[::-1]
c_aspect = n / T
sigma2 = float(np.median(sd) ** 2)
lam_minus = sigma2 * (1 - np.sqrt(c_aspect)) ** 2
lam_plus = sigma2 * (1 + np.sqrt(c_aspect)) ** 2

n_outliers = int((eigs > lam_plus).sum())
top10_share = float(eigs[:10].sum() / eigs.sum())

print(f"Aspect ratio c = n/T = {c_aspect:.3f}")
print(f"MP support:    [{lam_minus:.2e}, {lam_plus:.2e}]")
print(f"# eigenvalues above MP edge: {n_outliers}  (these are factor modes)")
print(f"Top 10 eigenvalue share:     {top10_share*100:.1f}%")
print(f"λ_max = {eigs[0]:.3e}   λ_max / λ_+ = {eigs[0]/lam_plus:.1f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
lam_grid = np.linspace(max(lam_minus, 1e-8), lam_plus, 400)
mp_density = np.sqrt(np.maximum((lam_plus - lam_grid) * (lam_grid - lam_minus), 0)) / (2 * np.pi * sigma2 * c_aspect * lam_grid)
ax.hist(eigs, bins=80, density=True, alpha=0.6, label="empirical")
ax.plot(lam_grid, mp_density, "r-", lw=2, label="Marchenko–Pastur")
ax.set_xlim(0, max(lam_plus * 1.5, np.quantile(eigs, 0.99)))
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("density")
ax.set_title("Bulk: empirical vs MP (truncated)")
ax.legend()

ax = axes[1]
ax.plot(np.arange(1, len(eigs) + 1), eigs, "o-", markersize=3)
ax.axhline(lam_plus, color="red", ls="--", label=fr"MP edge $\lambda_+={lam_plus:.2e}$")
ax.set_yscale("log")
ax.set_xlabel("index")
ax.set_ylabel("eigenvalue (log)")
ax.set_title(rf"Spectrum of $\hat\Sigma$ ({n_outliers} outliers above MP)")
ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "spectrum_vs_mp.png", dpi=120)
plt.show()

## Block 5, Factor extraction; residual spectrum

Strip out the top $r$ principal components ("market" + "sectors"). The residual covariance $\hat\Sigma - \sum_{j\le r}\lambda_j v_j v_j^\top$ should look much closer to MP. This estimates the **effective rank** of the systematic signal, call it $r^\star$. We will use a rank-$r^\star$ + diagonal model when stating the Phase 3a OGP theorem with dependent columns.

In [ ]:
evals, evecs = np.linalg.eigh(Sigma_hat)
evals = evals[::-1]
evecs = evecs[:, ::-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, r in zip(axes, [1, 5, 20]):
    Sigma_res = Sigma_hat - evecs[:, :r] @ np.diag(evals[:r]) @ evecs[:, :r].T
    eigs_res = np.linalg.eigvalsh(Sigma_res)[::-1]
    ax.hist(eigs_res, bins=80, density=True, alpha=0.6, label="empirical residual")
    ax.plot(lam_grid, mp_density, "r-", lw=2, label="MP")
    n_out_res = int((eigs_res > lam_plus).sum())
    ax.set_xlim(0, max(lam_plus * 1.5, np.quantile(eigs_res, 0.99)))
    ax.set_title(f"After removing top {r} factor(s):  {n_out_res} outliers above MP")
    ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "residual_spectrum.png", dpi=120)
plt.show()

## Block 6, Empirical RIP probe

Definition (standard compressive-sensing references): a matrix $X$ has the $s$-RIP with constant $\delta$ if for every $s$-sparse $\beta$
$$
(1-\delta)\|\beta\|_2^2 \;\le\; \|X\beta\|_2^2 \;\le\; (1+\delta)\|\beta\|_2^2.
$$
We probe with random unit-norm $k$-sparse $\beta$ and report the empirical $\delta_k = \max_{\text{sample}}\bigl|\|X\beta\|_2^2/T - 1\bigr|$ for the standardized matrix $Z/\sqrt T$. The the standard recovery theorem needs $\delta_{2s}<1/3$ for LASSO recovery, we will see in the report whether real data satisfies this for the cardinalities we care about ($k=5\dots 50$). It almost certainly *fails* for moderate $k$, which itself is a finding (real returns have too much structure for LASSO-style recovery to apply directly).

In [ ]:
Zn = Z / np.sqrt(T)
rng = np.random.default_rng(0)

def rip_probe(M: np.ndarray, k: int, n_samples: int = 1000, rng=None) -> np.ndarray:
    rng = rng or np.random.default_rng()
    T_, n_ = M.shape
    norms = np.empty(n_samples)
    for s in range(n_samples):
        idx = rng.choice(n_, size=k, replace=False)
        beta = rng.standard_normal(k)
        beta /= np.linalg.norm(beta)
        v = M[:, idx] @ beta
        norms[s] = float(v @ v)
    return norms

ks = [5, 10, 20, 50]
delta_summary = {}
fig, axes = plt.subplots(1, len(ks), figsize=(15, 3.5), sharey=True)
for ax, k in zip(axes, ks):
    norms = rip_probe(Zn, k, n_samples=1000, rng=rng)
    delta_emp = float(np.max(np.abs(norms - 1.0)))
    q05, q50, q95 = np.quantile(np.abs(norms - 1.0), [0.05, 0.5, 0.95])
    delta_summary[k] = {"max": delta_emp, "q95": float(q95), "median": float(q50)}
    ax.hist(norms, bins=40, color="slategray", alpha=0.85)
    ax.axvline(1.0, color="red", ls="--", lw=1.5)
    ax.set_title(rf"$k$={k}, $\delta^{{(\max)}}_k$={delta_emp:.2f}, $\delta^{{(q95)}}_k$={q95:.2f}")
    ax.set_xlabel(r"$\|X\beta\|^2/T$")
axes[0].set_ylabel("count")
plt.tight_layout()
plt.savefig(FIG_DIR / "rip_probe.png", dpi=120)
plt.show()

print("Empirical RIP (smaller is better; <1/3 needed for course Thm 1):")
for k, d in delta_summary.items():
    flag = "✓" if d["q95"] < 1/3 else "✗"
    print(f"  k={k:3d}  q95(δ)={d['q95']:.3f}  max(δ)={d['max']:.3f}  [{flag}]")

## Block 7, Extreme-event date stamping

Identify $(t,i)$ pairs with $|Z_{t,i}| > 5$ and group by date. Heavy clustering on a few days (typically Sep 2008, Mar 2020, Aug 2024) confirms time-series dependence, heavy-tail events are not i.i.d. across $t$. This justifies the **time-revelation Doob martingale** trick (standard concentration references) for backtest CIs in Phase 4b: revealing one day at a time gives a tighter Azuma-Hoeffding bound than treating all $T n$ entries independently.

In [ ]:
thresh = 5.0
extreme_mask = np.abs(Z) > thresh
extreme_per_day = extreme_mask.sum(axis=1)
extreme_df = pd.DataFrame({"date": dates, "n_extreme": extreme_per_day}).sort_values("n_extreme", ascending=False)
top_days = extreme_df.head(15)
print(f"Top 15 days with most |Z| > {thresh} (out of {n} stocks):")
print(top_days.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(dates, extreme_per_day, color="darkred", lw=0.8)
ax.set_ylabel(rf"# stocks with $|Z|>{thresh}$")
ax.set_title("Extreme-return events over time (clusters ⇒ time-series dependence)")
plt.tight_layout()
plt.savefig(FIG_DIR / "extreme_events.png", dpi=120)
plt.show()

## Final report

Save numeric summary and write the verdict that decides Phase 3 wording.

In [ ]:
report = {
    "T": int(T),
    "n": int(n),
    "date_start": str(dates[0].date()),
    "date_end": str(dates[-1].date()),
    "psi2": {
        "frac_finite": float(psi2_finite.mean()),
        "median_finite": float(np.median(psi2[psi2_finite])) if psi2_finite.any() else None,
        "q90_finite": float(np.quantile(psi2[psi2_finite], 0.9)) if psi2_finite.any() else None,
        "gaussian_ref": float(np.sqrt(8/3)),
    },
    "psi1": {
        "frac_finite": float(psi1_finite.mean()),
        "median_finite": float(np.median(psi1[psi1_finite])) if psi1_finite.any() else None,
        "q90_finite": float(np.quantile(psi1[psi1_finite], 0.9)) if psi1_finite.any() else None,
        "laplace_ref": float(1/np.log(2)),
    },
    "spectrum": {
        "aspect_ratio_n_over_T": float(c_aspect),
        "mp_lambda_minus": float(lam_minus),
        "mp_lambda_plus": float(lam_plus),
        "lambda_max": float(eigs[0]),
        "lambda_max_over_mp_plus": float(eigs[0] / lam_plus),
        "n_outliers_above_mp": int(n_outliers),
        "top10_eigenvalue_share": float(top10_share),
    },
    "rip": {str(k): v for k, v in delta_summary.items()},
}

with open(DATA_DIR / "diagnostics_phase1.json", "w") as f:
    json.dump(report, f, indent=2)

print("Saved", DATA_DIR / "diagnostics_phase1.json")
print(json.dumps(report, indent=2))

## Verdict

Read the printed report above and check:

1. **Sub-Gaussian vs sub-exponential.** If `psi2.frac_finite` is near 1, write the theory in the **sub-Gaussian / Hoeffding** language. If many $\psi_2$ values are infinite but `psi1.frac_finite` is high, switch to **Bernstein / sub-exponential** in the theorem statements. If $\psi_1$ also blows up for many stocks, we either truncate (winsorize) or move to a power-law tail bound.

2. **Factor structure.** `lambda_max_over_mp_plus` typically lands between 30 and 100 for S&P data, that is the *market factor*. `n_outliers_above_mp` between 5 and 30 is normal (sectors). This dictates the rank $r^\star$ in the dependent-Gaussian model used in Phase 3a.

3. **RIP.** Almost certainly $\delta_k$ exceeds $1/3$ for $k\ge 10$ on real data. That is a *finding*, not a bug, it means LASSO is not justified by the standard recovery theorem 1 directly, and we have to discuss this honestly in the writeup.

Once the verdict is in, paste the `diagnostics_phase1.json` here and we move on to Phase 2a (oracle + baselines).